In [4]:
import sys
!{sys.executable} -m pip install geopy requests scipy

   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/36.5 MB ? eta -:--:--
   - -------------------------------------- 1.0/36.5 MB 2.7 MB/s eta 0:00:14
   - -------------------------------------- 1.6/36.5 MB 3.0 MB/s eta 0:00:12
   -- ------------------------------------- 2.6/36.5 MB 3.5 MB/s eta 0:00:10
   ---- ----------------------------------- 3.9/36.5 MB 4.1 MB/s eta 0:00:08
   ------ --------------------------------- 5.8/36.5 MB 4.9 MB/s eta 0:00:07
   -------- ------------------------------- 7.9/36.5 MB 5.6 MB/s eta 0:00:06
   ---------- ----------------------------- 10.0/36.5 MB 6.2 MB/s eta 0:00:05
   ------------- -------------------------- 12.1/36.5 MB 6.8 MB/s eta 0:00:04
   ---------------- ----------------------- 14.7/36.5 MB 7.4 MB/s eta 0:00:03
   ------------------ --------------------- 17.3/36.5 MB 7.8 MB/s eta 0:00:03
   --------------------- ------------------ 19.7/36.5 MB 8.2 MB/s eta 0:00:03
   -----

In [ ]:
#Importing necessary libraries
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font='DejaVu Sans')
plt.rcParams.update({'figure.dpi': 130, 'axes.titlesize': 13, 'axes.labelsize': 11})

In [ ]:
#Data Acquisition and Cleaning
OPENAQ_URL = 'https://api.openaq.org/v3/measurements'

params = {
    'parameter_id': 2,
    'country_id': 'US',
    'limit': 10000,
    'date_from': '2023-01-01',
    'date_to':   '2024-01-01',
}

headers = {'X-API-Key': 'YOUR_OPENAQ_API_KEY'}

response = requests.get(OPENAQ_URL, params=params, headers=headers)
response.raise_for_status()

raw_aq = pd.DataFrame(response.json()['results'])

In [ ]:
raw_aq['latitude'] = raw_aq['coordinates'].apply(
    lambda x: x['latitude'] if isinstance(x, dict) else np.nan
)
raw_aq['longitude'] = raw_aq['coordinates'].apply(
    lambda x: x['longitude'] if isinstance(x, dict) else np.nan
)
raw_aq['timestamp'] = raw_aq['date'].apply(
    lambda x: x['utc'] if isinstance(x, dict) else np.nan
)

aq_df = raw_aq[['locationId', 'location', 'latitude', 'longitude', 'timestamp', 'value']].copy()
aq_df.rename(columns={'value': 'pm25_ug_m3'}, inplace=True)

In [ ]:
aq_df = aq_df[aq_df['pm25_ug_m3'] >= 0]
aq_df = aq_df[aq_df['pm25_ug_m3'] <= 500]

print(f'AQ records after cleaning : {len(aq_df):,}')

In [ ]:
census_df = pd.read_csv('acs_median_income_by_zip.csv', dtype={'zip_code': str})

census_df.rename(columns={
    'zip_code':                'zip_code',
    'median_household_income': 'median_income_usd'
}, inplace=True)

census_df.dropna(subset=['median_income_usd'], inplace=True)
census_df['zip_code'] = census_df['zip_code'].str.zfill(5)
census_df = census_df[census_df['median_income_usd'] > 0]

print(f'ZIP codes in Census data  : {len(census_df):,}')

In [ ]:
#Data Integration
geolocator = Nominatim(user_agent='urban_pollution_study_v1')
geocode    = RateLimiter(geolocator.reverse, min_delay_seconds=1)

def get_zip(lat, lon):
    try:
        location = geocode(f'{lat}, {lon}', exactly_one=True, language='en')
        if location and 'postcode' in location.raw.get('address', {}):
            return str(location.raw['address']['postcode']).zfill(5)[:5]
    except Exception:
        pass
    return None

unique_sensors = aq_df[['locationId', 'latitude', 'longitude']].drop_duplicates()

unique_sensors['zip_code'] = unique_sensors.apply(
    lambda row: get_zip(row['latitude'], row['longitude']), axis=1
)

aq_df = aq_df.merge(unique_sensors[['locationId', 'zip_code']], on='locationId', how='left')

In [ ]:
zip_agg = (
    aq_df.groupby('zip_code')['pm25_ug_m3']
    .agg(mean_pm25='mean', sensor_count='count', std_pm25='std')
    .reset_index()
)

unified_df = zip_agg.merge(
    census_df[['zip_code', 'median_income_usd', 'state']],
    on='zip_code',
    how='inner'
)

unified_df['income_quintile'] = pd.qcut(
    unified_df['median_income_usd'],
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

unified_df.dropna(subset=['mean_pm25', 'median_income_usd'], inplace=True)
unified_df.reset_index(drop=True, inplace=True)

unified_df.to_csv('unified_pollution_income.csv', index=False)

print(unified_df[['zip_code', 'mean_pm25', 'median_income_usd', 'income_quintile', 'state']].head(10).to_string())
print(f'\nFinal unified dataset shape: {unified_df.shape}')

In [ ]:
#Statistical Analysis
r, p_pearson = stats.pearsonr(unified_df['median_income_usd'], unified_df['mean_pm25'])

print('─' * 45)
print('  Pearson Correlation Analysis')
print('─' * 45)
print(f'  Correlation Coefficient (r) : {r:.4f}')
print(f'  p-value                     : {p_pearson:.4e}')
print(f'  Interpretation              : {"Statistically significant" if p_pearson < 0.05 else "Not significant"} (α = 0.05)')
print('─' * 45)

In [ ]:
quintile_groups = [
    unified_df[unified_df['income_quintile'] == q]['mean_pm25'].dropna()
    for q in range(1, 6)
]

f_stat, p_anova = stats.f_oneway(*quintile_groups)

print('─' * 45)
print('  One-Way ANOVA')
print('─' * 45)
print(f'  F-Statistic : {f_stat:.4f}')
print(f'  p-value     : {p_anova:.4e}')
print(f'  Result      : {"Significant difference across quintiles" if p_anova < 0.05 else "No significant difference"}')
print('─' * 45)

In [ ]:
summary = unified_df.groupby('income_quintile').agg(
    Mean_PM25=('mean_pm25', 'mean'),
    Median_PM25=('mean_pm25', 'median'),
    Std_PM25=('mean_pm25', 'std'),
    Mean_Income=('median_income_usd', 'mean'),
    ZIP_Count=('zip_code', 'count')
).round(2)

print('\nDescriptive Statistics by Income Quintile\n')
print(summary.to_string())

In [ ]:
#Visualization & Results
#Scatter Plot : Income vs Mean PM2.5
bar_colors = ['#d62728', '#ff7f0e', '#bcbd22', '#2ca02c', '#1f77b4']
quintile_palette = dict(zip(range(1, 6), bar_colors))

fig, ax = plt.subplots(figsize=(10, 6))

for q in range(1, 6):
    subset = unified_df[unified_df['income_quintile'] == q]
    ax.scatter(
        subset['median_income_usd'],
        subset['mean_pm25'],
        c=quintile_palette[q],
        alpha=0.55,
        s=28,
        label=f'Quintile {q}'
    )

m, b = np.polyfit(unified_df['median_income_usd'], unified_df['mean_pm25'], 1)
x_line = np.linspace(unified_df['median_income_usd'].min(), unified_df['median_income_usd'].max(), 300)
ax.plot(x_line, m * x_line + b, color='black', linewidth=1.8, linestyle='--', label='Regression Line')

ax.set_xlabel('Median Household Income (USD)')
ax.set_ylabel('Annual Mean PM2.5 (µg/m³)')
ax.set_title('Median Household Income vs. Annual Mean PM2.5 by ZIP Code')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(title='Income Quintile', framealpha=0.9)
ax.annotate(
    f'r = {r:.3f}  |  p = {p_pearson:.2e}',
    xy=(0.03, 0.93), xycoords='axes fraction',
    fontsize=10, color='black',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray')
)

plt.tight_layout()
plt.savefig('outputs/fig1_scatter_regression.png', bbox_inches='tight')
plt.show()

In [ ]:
#Bar Plot : Mean PM2.5 by Income Quintile
quintile_stats = unified_df.groupby('income_quintile').agg(
    mean_pm25=('mean_pm25', 'mean'),
    std_pm25=('mean_pm25', 'std')
).reset_index()

quintile_labels = ['Q1\n(Lowest)', 'Q2', 'Q3\n(Middle)', 'Q4', 'Q5\n(Highest)']

fig, ax = plt.subplots(figsize=(9, 6))

bars = ax.bar(
    quintile_labels,
    quintile_stats['mean_pm25'],
    yerr=quintile_stats['std_pm25'],
    color=bar_colors,
    edgecolor='white',
    linewidth=0.8,
    capsize=5,
    error_kw={'elinewidth': 1.4, 'ecolor': 'dimgray'}
)

for bar, val in zip(bars, quintile_stats['mean_pm25']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f'{val:.2f}',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

ax.set_xlabel('Income Quintile')
ax.set_ylabel('Mean Annual PM2.5 (µg/m³)')
ax.set_title('Mean PM2.5 Concentration by Income Quintile\n(Error bars = ±1 SD)')
ax.set_ylim(0, quintile_stats['mean_pm25'].max() + quintile_stats['std_pm25'].max() + 2)

plt.tight_layout()
plt.savefig('outputs/fig2_bar_quintile.png', bbox_inches='tight')
plt.show()

In [ ]:
#Box Plot : PM2.5 Distribution by Income Quintile
fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(
    data=unified_df,
    x='income_quintile',
    y='mean_pm25',
    palette=bar_colors,
    width=0.55,
    flierprops=dict(marker='o', markerfacecolor='gray', markersize=3, alpha=0.5),
    ax=ax
)

ax.set_xticklabels(['Q1 (Lowest)', 'Q2', 'Q3 (Middle)', 'Q4', 'Q5 (Highest)'])
ax.set_xlabel('Income Quintile')
ax.set_ylabel('Annual Mean PM2.5 (µg/m³)')
ax.set_title('Distribution of PM2.5 Across Income Quintiles')
ax.annotate(
    f'ANOVA: F = {f_stat:.2f}, p = {p_anova:.2e}',
    xy=(0.03, 0.94), xycoords='axes fraction',
    fontsize=10,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray')
)

plt.tight_layout()
plt.savefig('outputs/fig3_boxplot_quintile.png', bbox_inches='tight')
plt.show()

In [ ]:
#Violin Plot : PM2.5 Density by Income Quintile
fig, ax = plt.subplots(figsize=(10, 6))

sns.violinplot(
    data=unified_df,
    x='income_quintile',
    y='mean_pm25',
    palette=bar_colors,
    inner='quartile',
    linewidth=1.2,
    ax=ax
)

ax.set_xticklabels(['Q1 (Lowest)', 'Q2', 'Q3 (Middle)', 'Q4', 'Q5 (Highest)'])
ax.set_xlabel('Income Quintile')
ax.set_ylabel('Annual Mean PM2.5 (µg/m³)')
ax.set_title('PM2.5 Density Distribution by Income Quintile')

plt.tight_layout()
plt.savefig('outputs/fig4_violin_quintile.png', bbox_inches='tight')
plt.show()

In [ ]:
#Correlation Heatmap : All Numeric Features
numeric_cols = ['mean_pm25', 'std_pm25', 'sensor_count', 'median_income_usd', 'income_quintile']
corr_matrix = unified_df[numeric_cols].corr()

readable_labels = ['Mean PM2.5', 'PM2.5 Std Dev', 'Sensor Count', 'Median Income', 'Income Quintile']

fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    square=True,
    xticklabels=readable_labels,
    yticklabels=readable_labels,
    ax=ax
)

ax.set_title('Pearson Correlation Heatmap — All Numeric Features')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig('outputs/fig5_correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
#Top 10 Most vs. Least Polluted ZIP Codes with Income Annotations
top10_high = unified_df.nlargest(10, 'mean_pm25')[['zip_code', 'mean_pm25', 'median_income_usd']]
top10_low  = unified_df.nsmallest(10, 'mean_pm25')[['zip_code', 'mean_pm25', 'median_income_usd']]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(top10_high['zip_code'].astype(str), top10_high['mean_pm25'], color='#d62728', edgecolor='white')
axes[0].set_xlabel('Mean PM2.5 (µg/m³)')
axes[0].set_title('Top 10 Most Polluted ZIP Codes')
axes[0].invert_yaxis()
for i, (pm, inc) in enumerate(zip(top10_high['mean_pm25'], top10_high['median_income_usd'])):
    axes[0].text(pm + 0.1, i, f' ${inc:,.0f}', va='center', fontsize=8.5, color='dimgray')

axes[1].barh(top10_low['zip_code'].astype(str), top10_low['mean_pm25'], color='#1f77b4', edgecolor='white')
axes[1].set_xlabel('Mean PM2.5 (µg/m³)')
axes[1].set_title('Top 10 Least Polluted ZIP Codes')
axes[1].invert_yaxis()
for i, (pm, inc) in enumerate(zip(top10_low['mean_pm25'], top10_low['median_income_usd'])):
    axes[1].text(pm + 0.02, i, f' ${inc:,.0f}', va='center', fontsize=8.5, color='dimgray')

fig.suptitle('Most vs. Least Polluted ZIP Codes with Median Income Annotations', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('outputs/fig6_top10_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
#Disparity Gap Analysis : Q1 vs. Q5
q1_mean = unified_df[unified_df['income_quintile'] == 1]['mean_pm25'].mean()
q5_mean = unified_df[unified_df['income_quintile'] == 5]['mean_pm25'].mean()
gap_abs = q1_mean - q5_mean
gap_pct = (gap_abs / q5_mean) * 100

fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(
    ['Q1 — Lowest Income', 'Q5 — Highest Income'],
    [q1_mean, q5_mean],
    color=['#d62728', '#1f77b4'],
    width=0.45,
    edgecolor='white'
)

for bar, val in zip(bars, [q1_mean, q5_mean]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.15,
        f'{val:.2f} µg/m³',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax.annotate(
    f'Disparity Gap: {gap_abs:.2f} µg/m³  ({gap_pct:.1f}% higher in Q1)',
    xy=(0.5, 0.92), xycoords='axes fraction', ha='center',
    fontsize=10,
    bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', edgecolor='gray')
)

ax.set_ylabel('Mean Annual PM2.5 (µg/m³)')
ax.set_title('PM2.5 Pollution Disparity:\nLowest vs. Highest Income Quintile')
ax.set_ylim(0, max(q1_mean, q5_mean) * 1.3)

plt.tight_layout()
plt.savefig('outputs/fig7_disparity_gap.png', bbox_inches='tight')
plt.show()

print(f'Q1 Mean PM2.5 : {q1_mean:.3f} µg/m³')
print(f'Q5 Mean PM2.5 : {q5_mean:.3f} µg/m³')
print(f'Absolute Gap  : {gap_abs:.3f} µg/m³')
print(f'Relative Gap  : {gap_pct:.1f}%')